# Celeb-DF v2 Deepfake Detection — EfficientNet-B0

This notebook downloads Celeb-DF v2, creates train/validation/test splits, extracts video frames, trains EfficientNet-B0, and **saves a resumable checkpoint to Google Drive after every epoch**.

### Important
- Mount Google Drive first. Checkpoints, CSV manifests, and extracted frames are stored there so they survive a Colab runtime reset.
- If training is interrupted, rerun the notebook from the top. The training cell resumes from the latest completed epoch.
- The dataset itself may need to be downloaded again after a runtime reset, but existing extracted frames in Drive are reused.


## 1. Mount Google Drive and set persistent paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Deepfake_Project"
FRAME_ROOT = os.path.join(PROJECT_DIR, "celebdf_frames")
MANIFEST_DIR = os.path.join(PROJECT_DIR, "manifests")
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, "last_checkpoint.pth")
BEST_MODEL_PATH = os.path.join(PROJECT_DIR, "best_model.pth")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(FRAME_ROOT, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Frames folder:", FRAME_ROOT)


## 2. Install and download Celeb-DF v2

In [ ]:
!pip -q install kagglehub opencv-python-headless scikit-learn matplotlib tqdm

import kagglehub

dataset_path = kagglehub.dataset_download("reubensuju/celeb-df-v2")
print("Dataset downloaded to:", dataset_path)

print("\nTop-level contents:")
for item in os.listdir(dataset_path):
    full_path = os.path.join(dataset_path, item)
    print(item, "—", "folder" if os.path.isdir(full_path) else "file")


## 3. Build the video dataframe

In [ ]:
import pandas as pd

real_folders = ["YouTube-real", "Celeb-real"]
fake_folders = ["Celeb-synthesis"]

records = []

for folder in real_folders:
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Required folder not found: {folder_path}")

    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".mp4", ".avi", ".mov")):
            records.append({
                "path": os.path.join(folder_path, filename),
                "video": filename,
                "label": 0,
                "class": "REAL"
            })

for folder in fake_folders:
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Required folder not found: {folder_path}")

    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".mp4", ".avi", ".mov")):
            records.append({
                "path": os.path.join(folder_path, filename),
                "video": filename,
                "label": 1,
                "class": "FAKE"
            })

df = pd.DataFrame(records)

print("Total videos:", len(df))
print("\nClass counts:")
print(df["class"].value_counts())
display(df.head())


## 4. Create train, validation, and official test splits

In [ ]:
from sklearn.model_selection import train_test_split

test_list_path = os.path.join(dataset_path, "List_of_testing_videos.txt")
if not os.path.exists(test_list_path):
    raise FileNotFoundError(f"Official test list not found: {test_list_path}")

with open(test_list_path, "r") as f:
    test_relpaths = {
        line.strip().split(maxsplit=1)[1].replace("\\", "/")
        for line in f
        if line.strip() and len(line.strip().split(maxsplit=1)) == 2
    }

df["relative_path"] = df["path"].apply(
    lambda p: os.path.relpath(p, dataset_path).replace("\\", "/")
)

df["split"] = "remaining"
df.loc[df["relative_path"].isin(test_relpaths), "split"] = "test"

matched = (df["split"] == "test").sum()
print("Official test videos matched:", matched)
print("Test-list entries:", len(test_relpaths))

if matched == 0:
    raise ValueError(
        "No official test-list entries matched the discovered videos. "
        "Check the dataset folder structure and test-list path format."
    )

remaining = df[df["split"] == "remaining"].copy()

train_df, val_df = train_test_split(
    remaining,
    test_size=0.15,
    random_state=42,
    stratify=remaining["label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = df[df["split"] == "test"].copy()

train_df["split"] = "train"
val_df["split"] = "val"

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}: {len(part)}")
    print(part["class"].value_counts())

train_csv = os.path.join(MANIFEST_DIR, "train_videos.csv")
val_csv = os.path.join(MANIFEST_DIR, "val_videos.csv")
test_csv = os.path.join(MANIFEST_DIR, "test_videos.csv")

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("\nSplit manifests saved in:", MANIFEST_DIR)


## 5. Extract frames (skip existing frames)

Five frames are sampled evenly from each video. Frames are stored in Google Drive. If a video's expected frame files already exist, extraction is skipped, which saves time after a runtime restart.

In [ ]:
import cv2
import numpy as np
from tqdm.auto import tqdm

FRAMES_PER_VIDEO = 5
VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov")

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    for _, row in tqdm(
        split_df.iterrows(),
        total=len(split_df),
        desc=f"Extracting {split_name}"
    ):
        video_path = row["path"]
        label_name = row["class"]

        output_dir = os.path.join(FRAME_ROOT, split_name, label_name)
        os.makedirs(output_dir, exist_ok=True)

        video_id = os.path.splitext(os.path.basename(video_path))[0]
        expected_count = min(
            FRAMES_PER_VIDEO,
            max(0, int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT)))
        )
        expected_paths = [
            os.path.join(output_dir, f"{video_id}_{i}.jpg")
            for i in range(expected_count)
        ]

        if expected_paths and all(os.path.exists(p) for p in expected_paths):
            continue

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("Could not open:", video_path)
            cap.release()
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            continue

        frame_indices = np.linspace(
            0, total_frames - 1,
            min(FRAMES_PER_VIDEO, total_frames),
            dtype=int
        )

        for i, frame_idx in enumerate(frame_indices):
            output_path = os.path.join(output_dir, f"{video_id}_{i}.jpg")
            if os.path.exists(output_path):
                continue

            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            success, frame = cap.read()
            if not success:
                continue

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))

            cv2.imwrite(
                output_path,
                cv2.cvtColor(frame, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, 90]
            )

        cap.release()

print("Frame extraction complete!")


## 6. Verify extracted frames

In [ ]:
for split_name in ["train", "val", "test"]:
    for class_name in ["FAKE", "REAL"]:
        folder = os.path.join(FRAME_ROOT, split_name, class_name)
        count = (
            len([f for f in os.listdir(folder) if f.lower().endswith(".jpg")])
            if os.path.isdir(folder) else 0
        )
        print(f"{split_name:5} | {class_name:4} | {count} frames")


## 7. Create datasets and data loaders

In [ ]:
import torch
import torch.nn as nn
import copy
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_ds = datasets.ImageFolder(
    os.path.join(FRAME_ROOT, "train"),
    transform=train_transform
)
val_ds = datasets.ImageFolder(
    os.path.join(FRAME_ROOT, "val"),
    transform=eval_transform
)

print("Class mapping:", train_ds.class_to_idx)
print("Training frames:", len(train_ds))
print("Validation frames:", len(val_ds))

if len(train_ds) == 0 or len(val_ds) == 0:
    raise ValueError("Training or validation dataset is empty. Check frame extraction.")

# ImageFolder sorts alphabetically: FAKE=0, REAL=1
class_counts = np.bincount(train_ds.targets)
class_weights = len(train_ds) / (len(class_counts) * class_counts)
class_weights = torch.tensor(
    class_weights, dtype=torch.float32, device=device
)
print("Class counts [FAKE, REAL]:", class_counts)
print("Class weights:", class_weights.tolist())

# num_workers=0 is generally more reliable in Colab notebooks
train_loader = DataLoader(
    train_ds, batch_size=32, shuffle=True,
    num_workers=0, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False,
    num_workers=0, pin_memory=(device.type == "cuda")
)


## 8. Train EfficientNet-B0 with automatic checkpoint resume

**This is the key section.** A `last_checkpoint.pth` file is written to Drive after every completed epoch. If training is interrupted, rerun this cell after running the setup and data-loader cells. It loads the latest model weights, optimizer state, epoch number, and best validation loss.

The best validation model is saved separately as `best_model.pth`.

In [ ]:
# Build the same EfficientNet-B0 architecture each time
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 2
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-4
)

EPOCHS = 10
start_epoch = 0
best_val_loss = float("inf")

# Load the latest checkpoint, if one exists
if os.path.exists(CHECKPOINT_PATH):
    print("Previous checkpoint found. Loading...")

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device
    )

    # Safety check: make sure the saved class order matches this dataset
    saved_class_to_idx = checkpoint.get("class_to_idx")
    if saved_class_to_idx != train_ds.class_to_idx:
        raise ValueError(
            f"Class mapping mismatch. Saved: {saved_class_to_idx}, "
            f"current: {train_ds.class_to_idx}"
        )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]

    print("Checkpoint loaded!")
    print("Last completed epoch:", checkpoint["epoch"] + 1)
    print("Next epoch:", start_epoch + 1)

else:
    print("No checkpoint found. Starting from epoch 1.")

if start_epoch >= EPOCHS:
    print(f"Training already reached {EPOCHS} epochs.")
else:
    for epoch in range(start_epoch, EPOCHS):
        # ---------------- TRAINING ----------------
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # ---------------- VALIDATION ----------------
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        # Save best model when validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save({
                "model_state_dict": model.state_dict(),
                "class_to_idx": train_ds.class_to_idx,
                "architecture": "efficientnet_b0",
                "best_val_loss": best_val_loss,
                "epoch": epoch
            }, BEST_MODEL_PATH)

            print("Best model updated and saved!")

        # Save latest resumable checkpoint after every completed epoch
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "class_to_idx": train_ds.class_to_idx,
            "architecture": "efficientnet_b0"
        }, CHECKPOINT_PATH)

        print("Latest checkpoint saved to Drive.")

print("\nTraining cell finished.")
print("Latest checkpoint:", CHECKPOINT_PATH)
print("Best model:", BEST_MODEL_PATH)


## 9. Evaluate the best model on the official test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

test_ds = datasets.ImageFolder(
    os.path.join(FRAME_ROOT, "test"),
    transform=eval_transform
)

if len(test_ds) == 0:
    raise ValueError("Test dataset is empty. Check test frame extraction.")

test_loader = DataLoader(
    test_ds, batch_size=32, shuffle=False, num_workers=0
)

# Load the best validation-loss model for final evaluation
best_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(1).cpu().numpy()

        all_preds.extend(predictions)
        all_labels.extend(labels.numpy())

fake_label = test_ds.class_to_idx["FAKE"]

print("Test Accuracy:", accuracy_score(all_labels, all_preds))
print("Fake Precision:", precision_score(
    all_labels, all_preds, pos_label=fake_label, zero_division=0
))
print("Fake Recall:", recall_score(
    all_labels, all_preds, pos_label=fake_label, zero_division=0
))
print("Fake F1 Score:", f1_score(
    all_labels, all_preds, pos_label=fake_label, zero_division=0
))

target_names = [
    name for name, idx in sorted(
        test_ds.class_to_idx.items(), key=lambda item: item[1]
    )
]

print("\nClassification Report:")
print(classification_report(
    all_labels, all_preds, target_names=target_names, zero_division=0
))

print("Confusion Matrix (rows=actual, columns=predicted):")
print(confusion_matrix(all_labels, all_preds))


## 10. Upload a video and predict REAL or FAKE

In [ ]:
from PIL import Image
from google.colab import files

# Load best model for inference
inference_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

inference_model = models.efficientnet_b0(weights=None)
inference_model.classifier[1] = nn.Linear(
    inference_model.classifier[1].in_features, 2
)
inference_model.load_state_dict(inference_checkpoint["model_state_dict"])
inference_model = inference_model.to(device)
inference_model.eval()

# Match the evaluation preprocessing used during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ImageFolder class order is FAKE=0, REAL=1
class_to_idx = inference_checkpoint["class_to_idx"]
fake_index = class_to_idx["FAKE"]
real_index = class_to_idx["REAL"]

print("Model loaded successfully!")
print("Class mapping:", class_to_idx)

def predict_video(video_path, num_frames=20):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        cap.release()
        raise ValueError("Could not open this video.")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError("Could not read frames from this video.")

    frame_indices = np.linspace(
        0, total_frames - 1,
        min(num_frames, total_frames),
        dtype=int
    )

    fake_probs = []

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame)
        image_tensor = inference_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = inference_model(image_tensor)
            probabilities = torch.softmax(logits, dim=1)

        fake_probs.append(probabilities[0, fake_index].item())

    cap.release()

    if not fake_probs:
        raise ValueError("No readable frames were found.")

    avg_fake = float(np.mean(fake_probs))
    avg_real = 1.0 - avg_fake

    print(f"Frames analyzed: {len(fake_probs)}")
    print(f"Average FAKE score: {avg_fake:.2%}")
    print(f"Average REAL score: {avg_real:.2%}")

    if avg_fake >= 0.5:
        print("\nPrediction: FAKE")
    else:
        print("\nPrediction: REAL")

    return avg_fake

uploaded = files.upload()

if uploaded:
    video_path = next(iter(uploaded.keys()))
    fake_score = predict_video(video_path)


## How to resume after a Colab disconnect

1. Reopen this notebook and connect to a new runtime.
2. Run the cells from the beginning so Drive is mounted, the dataset/dataframes are prepared, and the dataloaders are recreated.
3. The frame-extraction cell skips frames already present in Drive.
4. Run the training cell. It loads `last_checkpoint.pth` and continues from the next epoch.

**Note:** The checkpoint is saved at the end of each epoch. If the runtime stops halfway through an epoch, that incomplete epoch will be repeated. For a reliable resume, keep `EPOCHS` the same when continuing a run. This notebook saves model and optimizer state, but does not save random-number-generator state, so resumed augmentation/shuffling will not be identical.

The test-set metrics are frame-level metrics because each extracted frame is treated as an individual image. They should not be interpreted as video-level accuracy.